# Preprocessing steps

## Loading and normalizing the data

In [1]:
from datasets import load_dataset 
ds = load_dataset("bigcode/bigcodebench") 

In [2]:
split_name = list(ds.keys())[-1]
print("Using split:", split_name)
dataset_split = ds[split_name] 
print(dataset_split[0].keys())

Using split: v0.1.4
dict_keys(['task_id', 'complete_prompt', 'instruct_prompt', 'canonical_solution', 'code_prompt', 'test', 'entry_point', 'doc_struct', 'libs'])


In [8]:
import uuid
import json

def normalize(dataset_split):
    normalized = []
    for entry in dataset_split: 
        doc_struct_raw = entry.get("doc_struct", "{}")
        try:
            doc_struct = json.loads(doc_struct_raw)
        except json.JSONDecodeError:
            doc_struct = {}

        # Extract description
        description_list = doc_struct.get("description", [])
        description_text = " ".join(description_list)

        # Original code
        original_code = entry.get("canonical_solution", "")

        # Append part of complete_prompt until """
        complete_prompt = entry.get("complete_prompt", "")
        if '"""' in complete_prompt:
            snippet = complete_prompt.split('"""', 1)[0]
            snippet = snippet.rstrip()
            original_code = snippet + "\n" + original_code

        normalized.append({
            "id": str(uuid.uuid4()),
            "language": entry.get("language", "python"),
            "original_code": original_code,
            "test": [entry.get("test", "")],
            "description": description_text,
            "metadata": {
                "task_id": entry.get("task_id"),
                "libs": entry.get("libs", [])
            },
            "clones": []
        })
    return normalized

normalized_data = normalize(dataset_split)

# Quick check
import json
print(json.dumps(normalized_data[0], indent=2))


{
  "id": "864dbb01-9fbf-4f90-b950-d8716230702c",
  "language": "python",
  "original_code": "import itertools\nfrom random import shuffle\n\ndef task_func(numbers=list(range(1, 3))):\n    permutations = list(itertools.permutations(numbers))\n    sum_diffs = 0\n\n    for perm in permutations:\n        perm = list(perm)\n        shuffle(perm)\n        diffs = [abs(perm[i] - perm[i+1]) for i in range(len(perm)-1)]\n        sum_diffs += sum(diffs)\n\n    avg_sum_diffs = sum_diffs / len(permutations)\n    \n    return avg_sum_diffs",
  "test": [
    "import unittest\nfrom unittest.mock import patch\nfrom random import seed, shuffle\nimport itertools\nclass TestCases(unittest.TestCase):\n    def test_default_numbers(self):\n        # Test with default number range (1 to 10) to check that the result is a positive float.\n        result = task_func()\n        self.assertIsInstance(result, float)\n        self.assertGreater(result, 0)\n    def test_custom_list(self):\n        # Test with a cus

In [29]:
import json

with open("../dataset/bigcodebench_normalized.json", "w", encoding="utf-8") as f:
    json.dump(normalized_data, f, indent=2)

print("Normalized dataset saved as bigcodebench_normalized.json")

## Running tests

In [28]:
import unittest
def run_bigcodebench_test_with_details(entry):
    """
    Run code + tests from a dataset entry and return which tests failed.
    Works with unittest-style tests.
    """
    try:
        namespace = {}
        # Load the candidate solution
        exec(entry["original_code"], namespace)

        # Load the test code
        test_code = entry["test"][0]
        exec(test_code, namespace)

        # Collect test cases
        suite = unittest.TestSuite()
        for obj in namespace.values():
            if isinstance(obj, type) and issubclass(obj, unittest.TestCase):
                suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(obj))

        # Run tests
        result = unittest.TestResult()
        suite.run(result)

        if result.wasSuccessful():
            return {"all_passed": True, "failed_tests": []}
        else:
            failed = [str(test) for test, _ in result.failures + result.errors]
            return {"all_passed": False, "failed_tests": failed}

    except Exception as e:
        return {
            "all_passed": False,
            "failed_tests": [f"__error__: {str(e)}"]
        }


In [ ]:
import json
import os
def evaluate_dataset(normalized_data, output_file="../results/original_failed_tests.json"): 
    failed_by_id = {}

    for i, entry in enumerate(normalized_data):
        result = run_bigcodebench_test_with_details(entry)
        if not result["all_passed"]:
            failed_by_id[entry["id"]] = result["failed_tests"]

   
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    # Save to JSON
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(failed_by_id, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(failed_by_id)} failed test entries to {output_file}")

evaluate_dataset(normalized_data)


<string>:6: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
<string>:7: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
<string>:6: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
<string>:6: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
<string>:6: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
<string>:6: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
<string>:10: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
<string>:6: FutureWarn